# Hz QPO search in Fermi GBM

Replication of **Caballero-García et al. (2024), Table 2 (Fermi/nb row)** using Stingray Leahy PDS.
Paper uses XSPEC Whittle + BXA (`S²/dof`). This notebook uses Stingray **Merit/dof** (`Σ[(P−M)/M]²`). 


---

## How to run

1. **Kernel:** Python 3 with `numpy`, `scipy`, `astropy`, `matplotlib`, `pandas`, `ipywidgets`, and local **Stingray** (`../stingray/`).
2. **Data:** barycentred TTE file in `test-data/`.

---

## Notebook map

| § | Section | What it does |
|---|---------|--------------|
| **0** | Setup | Imports, plotting style, dashboard helpers |
| **1** | Configuration | File paths, time/energy defaults, paper reference values |
| **2** | Load data | Read GBM TTE, dead-time correction helpers |
| **3** | Fit engine | Light curve → Leahy PDS → nested models (a,b,c) |
| **4** | Step 1 — Light curve | Sliders: segment, energy, binning; refits all models |
| **5** | Step 2 — Compare | Nested-model table, overlay PDS, **pick final fit** |
| **6** | Step 3 — Final fit | Parameter table + component PDS for chosen model |
| **7** | Cross-spectrum (nb×n8) | Two-detector CS → same nested fit + S/N vs §6 |

---

## Analysis flow

```
Load TTE  →  select segment & energy  →  Leahy PDS (0.1–10 Hz)
     →  fit (a) PL+WN  →  (b) +1 Lor  →  (c) +2 Lor
     →  compare Merit/dof & nested p-values
     →  choose final model  →  report ν, FWHM, S/N
```

## Equations & statistics

| Quantity | Implementation |
|----------|----------------|
| Dead time | `n_corr = n_obs / (1 − n_obs·τ)` with τ = 2.6 µs |
| PDS | Stingray `Powerspectrum`, `norm='leahy'` → WN level = 2 |
| Uncertainty | `σ_P = √(2/m)` (Leahy) |
| Model | `P = N₀(f/f₀)^(−η₀) + Lor(ν₁) + Lor(ν₂) + 2` |
| Goodness of fit | **Merit** = `Σ[(Pᵢ−Mᵢ)/Mᵢ]²`; **Merit/dof** from Stingray |
| Nested test | `ΔD = D_simple − D_complex`; `p` from χ²(Δν) |

**Nested models:** (a) PL+WN → (b) +1 Lorentzian → (c) +2nd Lorentzian (default final fit).


---

## §0 — Setup (imports & helpers)

In [1]:
import sys, warnings
from io import BytesIO
from pathlib import Path

import matplotlib
matplotlib.use("Agg")          # figures go to widget Output, not inline
import matplotlib.pyplot as plt
plt.ioff()

import numpy as np
import pandas as pd
from astropy.io import fits
from astropy.modeling import models
from scipy import stats

warnings.filterwarnings("ignore", category=RuntimeWarning)

INK = "#1c1c1c"
plt.rcParams.update({
    "figure.dpi": 120, "font.size": 11, "axes.grid": False,
    "axes.edgecolor": INK, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": INK, "ytick.color": INK,
    "figure.facecolor": "white", "axes.facecolor": "white",
    "legend.frameon": False,
})

NB_DIR = Path(".").resolve()
sys.path.insert(0, str((NB_DIR.parent / "stingray").resolve()))
from stingray.events import EventList
from stingray.powerspectrum import Powerspectrum
from stingray.modeling import fit_powerspectrum

import ipywidgets as widgets
from ipywidgets import FloatSlider, HBox, ToggleButtons, VBox
from IPython.display import Image, display, clear_output


def show_in(out, fig):
    buf = BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", dpi=120)
    plt.close(fig)
    plt.close("all")
    out.clear_output(wait=True)
    out.append_display_data(Image(buf.getvalue()))


def fill_output(out, *items):
    out.clear_output(wait=True)
    for item in items:
        if item is not None and (not hasattr(item, "empty") or not item.empty):
            out.append_display_data(item)


def dash_teardown(state):
    for w in state.get("widgets", ()):
        try:
            w.unobserve_all()
            w.close()
        except Exception:
            pass
    if state.get("root") is not None:
        try:
            state["root"].close()
        except Exception:
            pass
    state.update(widgets=(), root=None, outs=None, gen=state.get("gen", 0) + 1)


def wire_guarded(state, slider, callback):
    gen = state["gen"]
    def wrapped(*args, **kwargs):
        if state.get("gen") != gen:
            return
        return callback(*args, **kwargs)
    slider.unobserve_all()
    slider.observe(wrapped, names="value")

# §6 widget handles (updated when §6 cell runs)
FINAL = dict(out_res=None, out_pds=None, hdr=None, s_flo=None, s_fhi=None, s_lit=None)
_OBS = {}  # token -> callback; avoids unobserve_all on shared s_model


def wire_observer(widget, callback, token):
    """Attach one named observer without removing other cells' listeners."""
    old = _OBS.pop(token, None)
    if old is not None:
        try:
            widget.unobserve(old, names="value")
        except Exception:
            pass
    _OBS[token] = callback
    widget.observe(callback, names="value")


def refresh_final_fit(idx=None):
    """Refresh §6; implemented when §6 cell runs (safe no-op before that)."""
    fn = globals().get("_refresh_final_fit_impl")
    if callable(fn):
        fn(idx)


print("§0 OK — imports and helpers ready")


§0 OK — imports and helpers ready


/Users/vsharma8/Desktop/Papers/Paper_231129C/0_QPO_Analysis/stingray/stingray/utils.py:54: UserWarning: The recommended numba package is not installed. Some functionality might be slower.
  warnings.warn(


---

## §1 — Configuration

**Edit here** to change defaults before running downstream cells.

| Parameter | Default | Meaning |
|-----------|---------|---------|
| `TMIN`, `TMAX` | 7.3 – 10.0 s | PDS segment (paper value) |
| `EMIN`, `EMAX` | 8 – 900 keV | Energy band |
| `DT` | 10 ms (50 Hz Nyquist) | Light-curve bin size |
| `FMIN`, `FMAX` | 0.1 – 50 Hz | Frequency range for **fitting** |
| `SELECTED_MODEL_IDX` | 2 → model (c) | Initial final-fit choice |


In [2]:
GRB_NAME, DETECTOR = "GRB 220910A", "nb"
TTE_FILE = NB_DIR / "test-data" / "glg_tte_nb_bn220910242_v00_b.fit"

TMIN, TMAX = 7.3, 10.0
EMIN, EMAX = 8.0, 900.0
DT = 0.010

FMIN = 0.1
FMAX = 50.0
FMIN_PLOT, FMAX_PLOT = 0.5, 20.0

WN_LEVEL = 2.0
GBM_DEADTIME = 2.6e-6
NU1_INIT, NU2_INIT = 3.1, 7.1

NESTED_MODEL_LABELS = ["(a) PL + WN", "(b) PL + 1 Lor + WN", "(c) PL + 2 Lor + WN"]
SELECTED_MODEL_IDX = 2

SHOW_LIT_COMPARE = True
PAPER_FERMI = dict(
    eta0=5.0, N0=2.5e3,
    nu1=3.1, fwhm1=0.5, N_lor1=110,
    nu2=7.1, fwhm2=0.6, N_lor2=50,
    chi2_red=1.67, dof=19,
    sig1=(4, 1), sig2=(5, 2.4),
)
LITERATURE_REF = PAPER_FERMI if SHOW_LIT_COMPARE else None

LC_VIEW = (-10.0, 100.0)
SLIDER_STYLE = {"description_width": "initial"}
PDS_COLORS = dict(data="#000", total="#222", pl="#9B6B5C", lor1="#B84A62", lor2="#3A6EA5", wn="#8E9AAF")
CMP_COLORS = ["#7A7A7A", "#B84A62", "#3A6EA5"]

print(f"§1 OK — {TTE_FILE.name}")
print(f"  segment: {TMIN}–{TMAX} s | {EMIN:.0f}–{EMAX:.0f} keV | dt={DT*1e3:g} ms | fit {FMIN}–{FMAX} Hz")


§1 OK — glg_tte_nb_bn220910242_v00_b.fit
  segment: 7.3–10.0 s | 8–900 keV | dt=10 ms | fit 0.1–50.0 Hz


---

## §2 — Load GBM TTE data

Reads the barycentred **TTE** FITS file.

**Functions:** `load_gbm_tte`, `select_events`, `deadtime_correct`

**Output:** total event count in the file.


In [3]:
def load_gbm_tte(path):
    with fits.open(path) as hdul:
        trig = float(hdul["PRIMARY"].header["TRIGTIME"])
        eb, ev = hdul["EBOUNDS"].data, hdul["EVENTS"].data
    return ev["TIME"] - trig, ev["PHA"], eb["E_MIN"], eb["E_MAX"]


def select_events(t, pha, e_min, e_max, t0, t1, emin, emax):
    ch = np.where((e_min >= emin) & (e_max <= emax))[0]
    return t[np.isin(pha, ch) & (t >= t0) & (t < t1)]


def deadtime_correct(counts):
    n = np.asarray(counts, float)
    with np.errstate(divide="ignore", invalid="ignore"):
        out = n / (1.0 - n * GBM_DEADTIME)
    return np.where(np.isfinite(out), out, 0.0)


times, pha, e_min_ch, e_max_ch = load_gbm_tte(TTE_FILE)
lc = lc_full = ps = None
freq = power = fit_mask = leahy_err = None
nested_fits = fit = None

print(f"§2 OK — loaded {len(times):,} events from {TTE_FILE.name}")


§2 OK — loaded 654,118 events from glg_tte_nb_bn220910242_v00_b.fit


---

## §3 — Build PDS & fit nested models

**Pipeline:**
1. Select events in time/energy window
2. Dead-time-corrected light curve → Leahy PDS
3. Fit models (a), (b), (c) on 0.1–10 Hz
4. Set active fit to `SELECTED_MODEL_IDX`


In [4]:
def build_lc(t0, t1, emin, emax, dt):
    t0, t1 = sorted(map(float, (t0, t1)))
    emin, emax = sorted(map(float, (emin, emax)))
    dt = float(dt)
    if t1 <= t0:
        t1 = t0 + max(dt, 1e-4)
    sel = select_events(times, pha, e_min_ch, e_max_ch, t0, t1, emin, emax)
    ev = EventList(time=sel, gti=np.array([[t0, t1]]))
    lc_out = ev.to_lc(dt=dt, tstart=t0, tseg=t1 - t0)
    lc_out.counts = deadtime_correct(lc_out.counts)
    return lc_out, len(sel)


def peak_seeds(f, p):
    seeds = []
    for anchor in (NU1_INIT, NU2_INIT):
        m = (f >= anchor - 1.0) & (f <= anchor + 1.0)
        seeds.append(float(f[m][np.argmax(p[m])]) if m.any() else anchor)
    return seeds


def make_model(n_lor, f_ref, nu_seeds):
    pl = models.PowerLaw1D(PAPER_FERMI["N0"], x_0=f_ref, alpha=PAPER_FERMI["eta0"])
    pl.x_0.fixed = True
    wn = models.PowerLaw1D(WN_LEVEL, x_0=1.0, alpha=0.0)
    wn.x_0.fixed = wn.alpha.fixed = True
    m = pl
    for i in range(n_lor):
        lor = models.Lorentz1D(
            [PAPER_FERMI["N_lor1"], PAPER_FERMI["N_lor2"]][i],
            nu_seeds[i],
            [PAPER_FERMI["fwhm1"], PAPER_FERMI["fwhm2"]][i],
        )
        lor.x_0.bounds = [(2.0, 5.5), (5.5, 10.0)][i]
        lor.fwhm.bounds = (0.05, 3.0)
        lor.amplitude.bounds = (0, None)
        m = m + lor
    return m + wn


def free_params(model):
    return [v for v, fix, tie in zip(model.parameters, model.fixed.values(), model.tied.values())
            if not fix and not tie]


def fit_model(n_lor, nu_seeds, label):
    f, p = freq[fit_mask], power[fit_mask]
    model = make_model(n_lor, float(f.min()), nu_seeds[: max(n_lor, 1)])
    ps_fit = Powerspectrum()
    ps_fit.freq, ps_fit.power, ps_fit.m = f, p, ps.m
    ps_fit.df, ps_fit.norm = ps.df, "leahy"
    parest, res = fit_powerspectrum(ps_fit, model, free_params(model))
    mfit = parest.lpost.model
    noise = np.sqrt(2.0 / ps.m)
    out = dict(label=label, n_lor=n_lor, model=mfit,
               merit=res.merit, merit_red=res.merit / res.dof, dof=int(res.dof),
               deviance=res.deviance, eta0=mfit.alpha_0.value, N0=mfit.amplitude_0.value)
    for i in range(n_lor):
        out[f"nu{i+1}"] = getattr(mfit, f"x_0_{i+1}").value
        out[f"fwhm{i+1}"] = getattr(mfit, f"fwhm_{i+1}").value
        out[f"N_lor{i+1}"] = getattr(mfit, f"amplitude_{i+1}").value
        out[f"sig{i+1}"] = out[f"N_lor{i+1}"] / noise
        out[f"sig{i+1}_trials"] = out[f"sig{i+1}"] / np.sqrt(fit_mask.sum())
    return out


def pds_band(f_lo, f_hi):
    f_lo, f_hi = sorted(map(float, (f_lo, f_hi)))
    f_hi = min(f_hi, float(freq.max()))
    f_lo = max(f_lo, float(freq[freq > 0].min()))
    mask = (freq >= f_lo) & (freq <= f_hi)
    return mask, np.geomspace(f_lo, f_hi, 600)


def active_model_idx():
    if "s_model" in globals():
        return int(s_model.value)
    return int(SELECTED_MODEL_IDX)


def set_active_fit(idx=None):
    global fit, SELECTED_MODEL_IDX
    if not nested_fits:
        return fit
    idx = int(np.clip(idx if idx is not None else active_model_idx(), 0, len(nested_fits) - 1))
    SELECTED_MODEL_IDX = idx
    fit = nested_fits[idx]
    return fit


def fit_all_models():
    global nested_fits
    seeds = peak_seeds(freq[fit_mask], power[fit_mask])
    nested_fits = [fit_model(n, seeds, NESTED_MODEL_LABELS[n]) for n in range(3)]
    return set_active_fit(active_model_idx())


def rebuild_analysis(t0, t1, emin, emax, dt, verbose=True):
    global TMIN, TMAX, EMIN, EMAX, DT, lc, lc_full, ps, freq, power, fit_mask, leahy_err
    t0, t1 = sorted(map(float, (t0, t1)))
    emin, emax = sorted(map(float, (emin, emax)))
    dt = float(np.clip(dt, 1e-4, 0.1))
    TMIN, TMAX, EMIN, EMAX, DT = t0, t1, emin, emax, dt
    lc, n_ev = build_lc(t0, t1, emin, emax, dt)
    sel_w = select_events(times, pha, e_min_ch, e_max_ch, LC_VIEW[0], LC_VIEW[1], emin, emax)
    lc_full = EventList(time=sel_w, gti=np.array([[LC_VIEW[0], LC_VIEW[1]]])).to_lc(
        dt=dt, tstart=LC_VIEW[0], tseg=LC_VIEW[1] - LC_VIEW[0])
    lc_full.counts = deadtime_correct(lc_full.counts)
    ps = Powerspectrum.from_lightcurve(lc, norm="leahy")
    freq, power = ps.freq[1:], ps.power[1:]
    fit_mask = (freq >= FMIN) & (freq <= FMAX)
    leahy_err = np.sqrt(2.0 / ps.m)
    if fit_mask.sum() >= 5:
        fit_all_models()
    if verbose and fit:
        print(f"§3 fit — Events={n_ev:,} | m={ps.m:.0f} | bins={fit_mask.sum()} | dt={dt*1e3:g} ms")
        _nu = " ".join(f"nu{k}={fit[f'nu{k}']:.2f} Hz" for k in (1, 2) if f"nu{k}" in fit)
        print(f"  {fit['label']}: Merit/dof={fit['merit_red']:.2f}" + (f" | {_nu}" if _nu else ""))


rebuild_analysis(TMIN, TMAX, EMIN, EMAX, DT)


§3 fit — Events=11,565 | m=1 | bins=133 | dt=10 ms
  (c) PL + 2 Lor + WN: Merit/dof=0.93 | nu1=3.27 Hz nu2=6.99 Hz


---

## §4 — Step 1: Light curve & segment controls

| Control | Action |
|---------|--------|
| **t min / t max** | PDS segment → **refits** all models |
| **E min / E max** | Energy band → refit |
| **dt (ms)** | Bin size → refit |
| **LC min / max** | Zoom x-axis only |

Gold band = PDS segment used for §3–§6.


In [5]:
def plot_lc(out, v_lo, v_hi):
    v_lo, v_hi = sorted(map(float, (v_lo, v_hi)))
    fig, ax = plt.subplots(figsize=(10, 3.4))
    ax.axvspan(TMIN, TMAX, color="#F4D35E", alpha=0.35, label=f"PDS segment ({TMIN:.1f}-{TMAX:.1f} s)")
    ax.step(lc_full.time, lc_full.counts, where="mid", color="#2E86AB", lw=1.2)
    ax.set(xlim=(v_lo, v_hi), xlabel="T - T0 (s)", ylabel=f"Counts / {DT*1e3:g} ms",
           title=f"{GRB_NAME} GBM-{DETECTOR.upper()} | {EMIN:.0f}-{EMAX:.0f} keV")
    ax.legend(fontsize=9)
    fig.tight_layout()
    show_in(out, fig)


_LC = globals().setdefault("_LC", dict(gen=0))
dash_teardown(_LC)
out_lc = widgets.Output()

s_tmin = FloatSlider(value=TMIN, min=-10, max=100, step=0.1, description="t min (s)",
                     continuous_update=False, style=SLIDER_STYLE)
s_tmax = FloatSlider(value=TMAX, min=-10, max=100, step=0.1, description="t max (s)",
                     continuous_update=False, style=SLIDER_STYLE)
s_emin = FloatSlider(value=EMIN, min=8, max=2000, step=1, description="E min (keV)",
                     continuous_update=False, style=SLIDER_STYLE)
s_emax = FloatSlider(value=EMAX, min=10, max=20000, step=10, description="E max (keV)",
                     continuous_update=False, style=SLIDER_STYLE)
s_dt = FloatSlider(value=DT*1e3, min=0.1, max=100, step=0.1, description="dt (ms)",
                   continuous_update=False, style=SLIDER_STYLE)
s_vlo = FloatSlider(value=LC_VIEW[0], min=LC_VIEW[0], max=LC_VIEW[1], step=0.1,
                    description="LC min", style=SLIDER_STYLE)
s_vhi = FloatSlider(value=LC_VIEW[1], min=LC_VIEW[0], max=LC_VIEW[1], step=0.1,
                    description="LC max", style=SLIDER_STYLE)


def refresh_lc(_=None):
    plot_lc(out_lc, s_vlo.value, s_vhi.value)


def refit_all(_=None):
    t0, t1, e0, e1 = s_tmin.value, s_tmax.value, s_emin.value, s_emax.value
    if t1 <= t0: t1 = t0 + 0.1
    if e1 <= e0: e1 = e0 + 1
    rebuild_analysis(t0, t1, e0, e1, s_dt.value / 1e3)
    refresh_lc()


_LC["widgets"] = (s_tmin, s_tmax, s_emin, s_emax, s_dt, s_vlo, s_vhi)
for s in (s_tmin, s_tmax, s_emin, s_emax, s_dt): wire_guarded(_LC, s, refit_all)
for s in (s_vlo, s_vhi): wire_guarded(_LC, s, refresh_lc)
_LC["root"] = VBox([
    widgets.HTML("<b>§4 — PDS segment & energy</b> (change → refit all models)"),
    HBox([s_tmin, s_tmax]), HBox([s_emin, s_emax]), s_dt,
    widgets.HTML("<b>Light curve</b> (gold = PDS segment)"),
    HBox([s_vlo, s_vhi]), out_lc,
])
clear_output(wait=True)
display(_LC["root"])
refresh_lc()


---

## §5 — Choose final model

Compare nested models and **pick (a), (b), or (c)** with the toggle buttons.

In [6]:
def model_table(active_idx=None):
    active_idx = active_model_idx() if active_idx is None else int(active_idx)
    rows = []
    for i, nf in enumerate(nested_fits or []):
        row = dict(model=nf["label"], Merit_dof=f"{nf['merit_red']:.2f}", dof=nf["dof"],
                   active="→" if i == active_idx else "")
        if i:
            dd = nested_fits[i - 1]["deviance"] - nf["deviance"]
            dnu = nested_fits[i - 1]["dof"] - nf["dof"]
            p = stats.chi2.sf(max(dd, 0), dnu) if dnu > 0 else np.nan
            row["test"] = f"({chr(96+i-1)})->({chr(96+i)})"
            row["p"] = f"{p:.3g}"
        else:
            row["test"] = row["p"] = "-"
        for k in (1, 2):
            row[f"nu{k}"] = f"{nf[f'nu{k}']:.2f}" if f"nu{k}" in nf else "-"
            row[f"S/N{k}"] = f"{nf[f'sig{k}']:.1f}" if f"sig{k}" in nf else "-"
        rows.append(row)
    return pd.DataFrame(rows) if rows else None


def plot_pds_compare(out, f_lo, f_hi, active_idx=None):
    if not nested_fits:
        return
    active_idx = active_model_idx() if active_idx is None else int(active_idx)
    mask, ff = pds_band(f_lo, f_hi)
    fig, ax = plt.subplots(figsize=(8, 4.8))
    ax.errorbar(freq[mask], power[mask], xerr=0.5 * ps.df, yerr=leahy_err, fmt="none", alpha=0.35)
    ax.plot(freq[mask], power[mask], "o", mfc="none", mec=PDS_COLORS["data"], ms=4, label="Data")
    y_all = [power[mask], [WN_LEVEL]]
    for i, nf in enumerate(nested_fits):
        y = nf["model"](ff)
        y_all.append(y[y > 0])
        sel = i == active_idx
        ax.plot(ff, y, color=CMP_COLORS[i], lw=2.4 if sel else 1.2,
                ls="-" if sel else "--",
                label=f"{nf['label']} ({nf['merit_red']:.2f})" + (" ★" if sel else ""))
    ax.axhline(WN_LEVEL, color=PDS_COLORS["wn"], ls=":", label="WN=2")
    yp = np.concatenate([np.asarray(a)[np.isfinite(a) & (np.asarray(a) > 0)] for a in y_all if np.size(a)])
    ax.set(xscale="log", yscale="log", xlim=(ff[0], ff[-1]),
           ylim=(max(0.8, yp.min() * 0.7), yp.max() * 1.2),
           xlabel="Frequency (Hz)", ylabel="Power (Leahy)",
           title=f"§5 compare | {TMIN}-{TMAX} s | selected: {nested_fits[active_idx]['label']}")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.12)
    fig.tight_layout()
    show_in(out, fig)


try:
    _keep_idx = int(s_model.value)
except Exception:
    _keep_idx = int(SELECTED_MODEL_IDX)

_CMP = globals().setdefault("_CMP", dict(gen=0))
dash_teardown(_CMP)
out_tbl, out_pds = widgets.Output(), widgets.Output()

_model_opts = [(f"({chr(97+i)}) {NESTED_MODEL_LABELS[i].split(')', 1)[-1].strip()}", i)
               for i in range(len(NESTED_MODEL_LABELS))]
s_model = ToggleButtons(options=_model_opts, value=_keep_idx,
                        description="Final fit", style={"description_width": "initial"})
s_flo = FloatSlider(value=FMIN_PLOT, min=0.1, max=50, step=0.1, description="f min (Hz)", style=SLIDER_STYLE)
s_fhi = FloatSlider(value=FMAX_PLOT, min=0.1, max=50, step=0.1, description="f max (Hz)", style=SLIDER_STYLE)


def refresh_cmp(_=None):
    idx = active_model_idx()
    fill_output(out_tbl, model_table(idx))
    plot_pds_compare(out_pds, s_flo.value, s_fhi.value, idx)


def pick_model(change):
    idx = int(change["new"])
    set_active_fit(idx)
    refresh_cmp()
    refresh_final_fit(idx)


_CMP["widgets"] = (s_model, s_flo, s_fhi)
wire_observer(s_model, pick_model, "cmp_pick")
for s in (s_flo, s_fhi):
    wire_guarded(_CMP, s, refresh_cmp)
_CMP["root"] = VBox([
    widgets.HTML("<b>§5 — Nested model comparison</b>"),
    out_tbl,
    widgets.HTML("<b>Overlay PDS</b> (★ = selected model)"),
    HBox([s_flo, s_fhi]), out_pds,
    widgets.HTML("<b>Choose final fit</b> (a / b / c) → then run or check §6 below"),
    s_model,
])
clear_output(wait=True)
display(_CMP["root"])
set_active_fit(_keep_idx)
refresh_cmp()
refresh_final_fit(_keep_idx)


---

## §6 — Final fit report

**Run after §5.** Shows the model selected by the toggle in §5.


In [7]:
def _fit_row(source, nf):
    row = dict(source=source, model=nf["label"], eta0=nf["eta0"],
               Merit_dof=f"{nf['merit_red']:.2f}", dof=nf["dof"])
    for k in (1, 2):
        if f"nu{k}" in nf:
            row[f"nu{k}"] = f"{nf[f'nu{k}']:.2f}"
            row[f"FWHM{k}"] = f"{nf[f'fwhm{k}']:.2f}"
            row[f"SN{k}"] = f"{nf[f'sig{k}']:.1f}, {nf[f'sig{k}_trials']:.1f}"
        else:
            row[f"nu{k}"] = row[f"FWHM{k}"] = row[f"SN{k}"] = "-"
    return row


def results_table(show_lit=False):
    if fit is None:
        return None
    rows = [_fit_row("Selected fit", fit)]
    ref = globals().get("LITERATURE_REF")
    if show_lit and ref:
        lit = dict(
            label="PL+2Lor+WN (lit.)", eta0=ref["eta0"],
            merit_red=ref["chi2_red"], dof=ref["dof"],
            nu1=ref["nu1"], fwhm1=ref["fwhm1"],
            sig1=ref["sig1"][0], sig1_trials=ref["sig1"][1],
            nu2=ref["nu2"], fwhm2=ref["fwhm2"],
            sig2=ref["sig2"][0], sig2_trials=ref["sig2"][1],
        )
        rows.append(_fit_row("Literature ref.", lit))
    return pd.DataFrame(rows)


def plot_pds_active(out, f_lo, f_hi):
    if fit is None:
        return
    mask, ff = pds_band(f_lo, f_hi)
    mfit = fit["model"]
    n_lor = fit["n_lor"]
    fig, ax = plt.subplots(figsize=(8, 4.8))
    ax.errorbar(freq[mask], power[mask], xerr=0.5 * ps.df, yerr=leahy_err, fmt="none", alpha=0.35)
    ax.plot(freq[mask], power[mask], "o", mfc="none", mec=PDS_COLORS["data"], ms=4, label="Data")
    pl = models.PowerLaw1D(mfit.amplitude_0.value, x_0=mfit.x_0_0.value, alpha=mfit.alpha_0.value)
    ax.plot(ff, pl(ff), ":", color=PDS_COLORS["pl"], label=f"PL eta0={fit['eta0']:.1f}")
    for i in range(n_lor):
        lor = models.Lorentz1D(getattr(mfit, f"amplitude_{i+1}").value,
                               getattr(mfit, f"x_0_{i+1}").value,
                               getattr(mfit, f"fwhm_{i+1}").value)
        ax.plot(ff, lor(ff), "--", color=PDS_COLORS[f"lor{i+1}"],
                label=f"Lor{i+1} {fit[f'nu{i+1}']:.2f} Hz")
    ax.plot(ff, mfit(ff), color=PDS_COLORS["total"], lw=2, label="Total")
    ax.axhline(WN_LEVEL, color=PDS_COLORS["wn"], ls=":", label="WN=2")
    y_all = [power[mask], mfit(freq[mask]), [WN_LEVEL]]
    yp = np.concatenate([np.asarray(a)[np.isfinite(a) & (np.asarray(a) > 0)] for a in y_all if np.size(a)])
    ax.set(xscale="log", yscale="log", xlim=(ff[0], ff[-1]),
           ylim=(max(0.8, yp.min() * 0.7), yp.max() * 1.2),
           xlabel="Frequency (Hz)", ylabel="Power (Leahy)",
           title=f"§6 final: {fit['label']} | Merit/dof={fit['merit_red']:.2f}")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.12)
    fig.tight_layout()
    show_in(out, fig)


def _refresh_final_fit_impl(idx=None):
    if idx is not None and not isinstance(idx, (int, np.integer)):
        idx = None
    idx = active_model_idx() if idx is None else int(idx)
    set_active_fit(idx)
    if FINAL.get("out_res") is None or fit is None:
        return
    FINAL["hdr"].value = (
        f"<b>§6 Final fit:</b> <code>{fit['label']}</code> "
        f"(Merit/dof={fit['merit_red']:.2f}, {fit['n_lor']} Lor)"
    )
    show_lit = FINAL["s_lit"].value if FINAL.get("s_lit") is not None else False
    fill_output(FINAL["out_res"], results_table(show_lit=show_lit))
    plot_pds_active(FINAL["out_pds"], FINAL["s_flo"].value, FINAL["s_fhi"].value)


globals()["_refresh_final_fit_impl"] = _refresh_final_fit_impl


def _act_on_model(change):
    _refresh_final_fit_impl(int(change["new"]))


_ACT = globals().setdefault("_ACT", dict(gen=0))
dash_teardown(_ACT)
out_res, out_pds = widgets.Output(), widgets.Output()
hdr = widgets.HTML("")

s_flo2 = FloatSlider(value=FMIN_PLOT, min=0.1, max=50, step=0.1, description="f min (Hz)", style=SLIDER_STYLE)
s_fhi2 = FloatSlider(value=FMAX_PLOT, min=0.1, max=50, step=0.1, description="f max (Hz)", style=SLIDER_STYLE)
s_lit = widgets.Checkbox(value=bool(LITERATURE_REF), description="Show literature ref.",
                         style=SLIDER_STYLE) if LITERATURE_REF else widgets.HTML("")

FINAL.update(out_res=out_res, out_pds=out_pds, hdr=hdr, s_flo=s_flo2, s_fhi=s_fhi2, s_lit=s_lit)

_ACT["widgets"] = (s_flo2, s_fhi2) + ((s_lit,) if LITERATURE_REF else ())
wire_guarded(_ACT, s_flo2, _refresh_final_fit_impl)
wire_guarded(_ACT, s_fhi2, _refresh_final_fit_impl)
if LITERATURE_REF:
    wire_guarded(_ACT, s_lit, _refresh_final_fit_impl)

_lit_row = s_lit if LITERATURE_REF else widgets.HTML("")
_ACT["root"] = VBox([
    hdr, out_res,
    widgets.HTML("<b>Component PDS</b>"),
    HBox([s_flo2, s_fhi2]), _lit_row, out_pds,
])
clear_output(wait=True)
display(_ACT["root"])

if "s_model" in globals():
    wire_observer(s_model, _act_on_model, "act_pick")

_refresh_final_fit_impl(active_model_idx())


---
## §7 — Cross-spectrum (NaI-b × NaI8)

**Goal:** Check QPO significance by combining two barycentred GBM detectors with the **same workflow as §3–§6**, but the spectrum is the **Leahy cross-spectrum** (|CS| vs *f*) from Stingray `AveragedCrossspectrum`.

| | Single detector (§3–§6) | Cross-spectrum (§7) |
|--|-------------------------|---------------------|
| Data | `glg_tte_nb_…_b.fit` | nb (interest) × n8 (reference) |
| Spectrum | Auto-PDS Leahy | \|cross power\| Leahy |
| Fit | PL + Lor + WN nested | Same models on \|CS\| |
| S/N | `N_lor / √(2/m)` | \|CS(ν)\| / √(2/m) at QPO + coherence |

Both TTE files: `gbm-data-barycorr/` (same time axis).

**Run order:** §7 fit → §7 LC → §7 compare → §7 final (mirrors §4–§6).


In [8]:
from stingray.crossspectrum import AveragedCrossspectrum
from stingray.powerspectrum import AveragedPowerspectrum

TTE_NB_XS = NB_DIR / "test-data" / "glg_tte_nb_bn220910242_v00_b.fit"
TTE_N8_XS = NB_DIR / "test-data" / "glg_tte_n8_bn220910242_v00_b.fit"

XS_SELECTED = 2
XS = dict(
    cs=None, pds_nb=None, pds_n8=None, lc_nb=None, lc_n8=None, lc_nb_full=None, lc_n8_full=None,
    freq=None, power=None, fit_mask=None, leahy_err=None, ps=None,
    nested_fits=None, fit=None, seg=None, n_nb=0, n_n8=0,
)


def _xs_load(path):
    with fits.open(path) as hdul:
        trig = float(hdul["PRIMARY"].header["TRIGTIME"])
        eb, ev = hdul["EBOUNDS"].data, hdul["EVENTS"].data
    return ev["TIME"] - trig, ev["PHA"], eb["E_MIN"], eb["E_MAX"]


times_xs_nb, pha_xs_nb, emin_xs_nb, emax_xs_nb = _xs_load(TTE_NB_XS)
times_xs_n8, pha_xs_n8, emin_xs_n8, emax_xs_n8 = _xs_load(TTE_N8_XS)


def build_xs_lc(times, pha, emin_ch, emax_ch, t0, t1, emin, emax, dt):
    t0, t1 = sorted(map(float, (t0, t1)))
    emin, emax = sorted(map(float, (emin, emax)))
    dt = float(dt)
    if t1 <= t0:
        t1 = t0 + max(dt, 1e-4)
    sel = select_events(times, pha, emin_ch, emax_ch, t0, t1, emin, emax)
    ev = EventList(time=sel, gti=np.array([[t0, t1]]))
    lc_out = ev.to_lc(dt=dt, tstart=t0, tseg=t1 - t0)
    lc_out.counts = deadtime_correct(lc_out.counts)
    return lc_out, len(sel)


def peak_seeds_xs(f, p):
    """Seed CS fits from §6 QPO frequencies when available."""
    gfit = globals().get("fit")
    if gfit is not None and "nu1" in gfit:
        seeds = [float(gfit["nu1"])]
        seeds.append(float(gfit["nu2"]) if "nu2" in gfit else NU2_INIT)
        return seeds
    return peak_seeds(f, p)


def fit_spectrum(f, p, ps_m, ps_df, fmin, fmax, labels=None, seeds=None):
    labels = labels or NESTED_MODEL_LABELS
    mask = (f >= fmin) & (f <= fmax)
    if mask.sum() < 5:
        return None, None
    seeds = peak_seeds_xs(f[mask], p[mask]) if seeds is None else seeds
    err = np.sqrt(2.0 / ps_m)

    def _one(n_lor, nu_seeds, label):
        fm = f[mask]
        model = make_model(n_lor, float(fm.min()), nu_seeds[: max(n_lor, 1)])
        ps_fit = Powerspectrum()
        ps_fit.freq, ps_fit.power, ps_fit.m = fm, p[mask], ps_m
        ps_fit.df, ps_fit.norm = ps_df, "leahy"
        parest, res = fit_powerspectrum(ps_fit, model, free_params(model))
        mfit = parest.lpost.model
        out = dict(label=label, n_lor=n_lor, model=mfit,
                   merit=res.merit, merit_red=res.merit / res.dof, dof=int(res.dof),
                   deviance=res.deviance, eta0=mfit.alpha_0.value, N0=mfit.amplitude_0.value)
        for i in range(n_lor):
            out[f"nu{i+1}"] = getattr(mfit, f"x_0_{i+1}").value
            out[f"fwhm{i+1}"] = getattr(mfit, f"fwhm_{i+1}").value
            out[f"N_lor{i+1}"] = getattr(mfit, f"amplitude_{i+1}").value
            out[f"sig{i+1}"] = out[f"N_lor{i+1}"] / err
            out[f"sig{i+1}_trials"] = out[f"sig{i+1}"] / np.sqrt(mask.sum())
        return out

    nfs = [_one(n, seeds, labels[n]) for n in range(3)]
    ctx = dict(freq=f, power=p, mask=mask, err=err, fmin=fmin, fmax=fmax)
    return nfs, ctx


def xs_pds_band(f_lo, f_hi):
    f_lo, f_hi = sorted(map(float, (f_lo, f_hi)))
    f = XS["freq"]
    f_hi = min(f_hi, float(f.max()))
    f_lo = max(f_lo, float(f[f > 0].min()))
    mask = (f >= f_lo) & (f <= f_hi)
    return mask, np.geomspace(f_lo, f_hi, 600)


def xs_active_idx():
    if "s_xs_model" in globals():
        return int(s_xs_model.value)
    return int(XS_SELECTED)


def xs_set_active(idx=None):
    if not XS.get("nested_fits"):
        XS["fit"] = None
        return None
    idx = int(np.clip(idx if idx is not None else xs_active_idx(), 0, len(XS["nested_fits"]) - 1))
    globals()["XS_SELECTED"] = idx
    XS["fit"] = XS["nested_fits"][idx]
    return XS["fit"]


def rebuild_xs(t0, t1, emin, emax, dt, verbose=True):
    global XS_TMIN, XS_TMAX, XS_EMIN, XS_EMAX, XS_DT
    t0, t1 = sorted(map(float, (t0, t1)))
    emin, emax = sorted(map(float, (emin, emax)))
    dt = float(np.clip(dt, 1e-4, 0.1))
    seg = t1 - t0
    XS_TMIN, XS_TMAX, XS_EMIN, XS_EMAX, XS_DT = t0, t1, emin, emax, dt

    lc_nb, n_nb = build_xs_lc(times_xs_nb, pha_xs_nb, emin_xs_nb, emax_xs_nb, t0, t1, emin, emax, dt)
    lc_n8, n_n8 = build_xs_lc(times_xs_n8, pha_xs_n8, emin_xs_n8, emax_xs_n8, t0, t1, emin, emax, dt)
    if n_nb < 10 or n_n8 < 10:
        return False

    cs = AveragedCrossspectrum.from_lightcurve(lc_nb, lc_n8, segment_size=seg, norm="leahy", silent=True)
    pds_nb = AveragedPowerspectrum.from_lightcurve(lc_nb, segment_size=seg, norm="leahy", silent=True)
    pds_n8 = AveragedPowerspectrum.from_lightcurve(lc_n8, segment_size=seg, norm="leahy", silent=True)

    f_cs, p_cs = cs.freq[cs.freq > 0], np.abs(cs.power[cs.freq > 0])
    nfs, ctx = fit_spectrum(f_cs, p_cs, cs.m, cs.df, FMIN, FMAX)
    if nfs is None:
        return False

    sel_nb = select_events(times_xs_nb, pha_xs_nb, emin_xs_nb, emax_xs_nb, LC_VIEW[0], LC_VIEW[1], emin, emax)
    sel_n8 = select_events(times_xs_n8, pha_xs_n8, emin_xs_n8, emax_xs_n8, LC_VIEW[0], LC_VIEW[1], emin, emax)
    lc_nb_full = EventList(time=sel_nb, gti=np.array([[LC_VIEW[0], LC_VIEW[1]]])).to_lc(
        dt=dt, tstart=LC_VIEW[0], tseg=LC_VIEW[1] - LC_VIEW[0])
    lc_n8_full = EventList(time=sel_n8, gti=np.array([[LC_VIEW[0], LC_VIEW[1]]])).to_lc(
        dt=dt, tstart=LC_VIEW[0], tseg=LC_VIEW[1] - LC_VIEW[0])
    lc_nb_full.counts = deadtime_correct(lc_nb_full.counts)
    lc_n8_full.counts = deadtime_correct(lc_n8_full.counts)

    ps_proxy = Powerspectrum()
    ps_proxy.m, ps_proxy.df, ps_proxy.norm = cs.m, cs.df, "leahy"

    XS.update(
        cs=cs, pds_nb=pds_nb, pds_n8=pds_n8, lc_nb=lc_nb, lc_n8=lc_n8,
        lc_nb_full=lc_nb_full, lc_n8_full=lc_n8_full,
        freq=f_cs, power=p_cs, fit_mask=ctx["mask"], leahy_err=ctx["err"], ps=ps_proxy,
        nested_fits=nfs, seg=seg, n_nb=n_nb, n_n8=n_n8,
    )
    xs_set_active(xs_active_idx())
    if verbose and XS["fit"]:
        xf = XS["fit"]
        _nu = " ".join(f"nu{k}={xf[f'nu{k}']:.2f} Hz" for k in (1, 2) if f"nu{k}" in xf)
        print(f"§7 fit — |CS| Leahy | nb={n_nb:,} n8={n_n8:,} | m={cs.m} | bins={ctx['mask'].sum()} | dt={dt*1e3:g} ms")
        print(f"  {xf['label']}: Merit/dof={xf['merit_red']:.2f}" + (f" | {_nu}" if _nu else ""))
    return True


def xs_cs_snr_at(nu_hz):
    """|CS| / Leahy noise at frequency nu."""
    cs = XS.get("cs")
    if cs is None:
        return np.nan
    i = int(np.argmin(np.abs(cs.freq - nu_hz)))
    return float(np.abs(cs.power[i]) / np.sqrt(2.0 / cs.m))


def xs_coherence_at(nu_hz):
    cs, pnb, pn8 = XS.get("cs"), XS.get("pds_nb"), XS.get("pds_n8")
    if cs is None:
        return np.nan
    i = int(np.argmin(np.abs(cs.freq - nu_hz)))
    p1, p2 = float(pnb.power[i]), float(pn8.power[i])
    if p1 <= 0 or p2 <= 0:
        return np.nan
    return float(np.abs(cs.power[i]) ** 2 / (p1 * p2))


def significance_table():
    """Compare §6 individual nb vs §7 cross-spectrum at fitted QPO frequencies."""
    rows = []
    xfit = XS.get("fit")
    if fit is None and xfit is None:
        return None
    for k in (1, 2):
        if xfit is None or f"nu{k}" not in xfit:
            continue
        nu = float(xfit[f"nu{k}"])
        row = dict(QPO=f"ν{k}", freq_Hz=f"{nu:.2f}")
        if fit is not None and f"nu{k}" in fit and abs(fit[f"nu{k}"] - nu) < 2.0:
            row["nb_S_N"] = f"{fit[f'sig{k}']:.1f}"
            row["nb_S_N_trials"] = f"{fit[f'sig{k}_trials']:.1f}"
        else:
            row["nb_S_N"] = row["nb_S_N_trials"] = "—"
        cs_sn = xs_cs_snr_at(nu)
        row["CS_S_N"] = f"{cs_sn:.1f}"
        row["coherence"] = f"{xs_coherence_at(nu):.3f}"
        if fit is not None and f"sig{k}" in fit and fit[f"sig{k}"] > 0:
            row["CS_vs_nb"] = f"{cs_sn / fit[f'sig{k}']:.2f}×"
        else:
            row["CS_vs_nb"] = "—"
        if xfit is not None and f"sig{k}" in xfit:
            row["CS_Lor_S_N"] = f"{xfit[f'sig{k}']:.1f}"
            if fit is not None and f"sig{k}" in fit and fit[f"sig{k}"] > 0:
                row["Lor_vs_nb"] = f"{xfit[f'sig{k}'] / fit[f'sig{k}']:.2f}×"
            else:
                row["Lor_vs_nb"] = "—"
        rows.append(row)
    return pd.DataFrame(rows) if rows else None


rebuild_xs(TMIN, TMAX, EMIN, EMAX, DT)
print(f"§7 engine OK — {TTE_NB_XS.name} × {TTE_N8_XS.name}")


/Users/vsharma8/Desktop/Papers/Paper_231129C/0_QPO_Analysis/stingray/stingray/fourier.py:1766: UserWarning: n_ave is below 30. Please note that the error bars on the quantities derived from the cross spectrum are only reliable for a large number of averaged powers.
  warnings.warn(


§7 fit — |CS| Leahy | nb=11,565 n8=10,228 | m=1 | bins=134 | dt=10 ms
  (c) PL + 2 Lor + WN: Merit/dof=0.54 | nu1=5.50 Hz nu2=7.14 Hz
§7 engine OK — glg_tte_nb_bn220910242_v00_b.fit × glg_tte_n8_bn220910242_v00_b.fit


### §7 Step 1 — Light curves & segment

In [9]:
def plot_xs_lcs(out, v_lo, v_hi):
    v_lo, v_hi = sorted(map(float, (v_lo, v_hi)))
    fig, ax = plt.subplots(figsize=(10, 3.4))
    ax.axvspan(XS_TMIN, XS_TMAX, color="#F4D35E", alpha=0.35,
               label=f"CS segment ({XS_TMIN:.1f}-{XS_TMAX:.1f} s)")
    ax.step(XS["lc_nb_full"].time, XS["lc_nb_full"].counts, where="mid", color="#B84A62", lw=1.1, label="NaI-b")
    ax2 = ax.twinx()
    ax2.step(XS["lc_n8_full"].time, XS["lc_n8_full"].counts, where="mid", color="#3A6EA5", lw=1.1, alpha=0.85, label="NaI8")
    ax.set(xlim=(v_lo, v_hi), xlabel="T − T₀ (s)", ylabel="Counts nb",
           title=f"§7 LC | {XS_EMIN:.0f}-{XS_EMAX:.0f} keV | nb={XS['n_nb']:,} n8={XS['n_n8']:,}")
    ax2.set_ylabel("Counts n8", color="#3A6EA5")
    ax.legend(loc="upper left", fontsize=8)
    ax2.legend(loc="upper right", fontsize=8)
    fig.tight_layout()
    show_in(out, fig)


_XS_LC = globals().setdefault("_XS_LC", dict(gen=0))
dash_teardown(_XS_LC)
out_xs_lc = widgets.Output()

s_xs_tmin = FloatSlider(value=TMIN, min=-10, max=100, step=0.1, description="t min (s)",
                        continuous_update=False, style=SLIDER_STYLE)
s_xs_tmax = FloatSlider(value=TMAX, min=-10, max=100, step=0.1, description="t max (s)",
                        continuous_update=False, style=SLIDER_STYLE)
s_xs_emin = FloatSlider(value=EMIN, min=8, max=2000, step=1, description="E min (keV)",
                        continuous_update=False, style=SLIDER_STYLE)
s_xs_emax = FloatSlider(value=EMAX, min=10, max=20000, step=10, description="E max (keV)",
                        continuous_update=False, style=SLIDER_STYLE)
s_xs_dt = FloatSlider(value=DT * 1e3, min=0.1, max=100, step=0.1, description="dt (ms)",
                      continuous_update=False, style=SLIDER_STYLE)
s_xs_vlo = FloatSlider(value=LC_VIEW[0], min=LC_VIEW[0], max=LC_VIEW[1], step=0.1,
                       description="LC min", style=SLIDER_STYLE)
s_xs_vhi = FloatSlider(value=LC_VIEW[1], min=LC_VIEW[0], max=LC_VIEW[1], step=0.1,
                       description="LC max", style=SLIDER_STYLE)


def refresh_xs_lc(_=None):
    plot_xs_lcs(out_xs_lc, s_xs_vlo.value, s_xs_vhi.value)


def refit_xs(_=None):
    t0, t1 = s_xs_tmin.value, s_xs_tmax.value
    e0, e1 = s_xs_emin.value, s_xs_emax.value
    if t1 <= t0: t1 = t0 + 0.1
    if e1 <= e0: e1 = e0 + 1
    if rebuild_xs(t0, t1, e0, e1, s_xs_dt.value / 1e3):
        refresh_xs_lc()
        if "refresh_xs_cmp" in globals():
            refresh_xs_cmp()
        fn = globals().get("refresh_xs_final")
        if callable(fn):
            fn()


_XS_LC["widgets"] = (s_xs_tmin, s_xs_tmax, s_xs_emin, s_xs_emax, s_xs_dt, s_xs_vlo, s_xs_vhi)
for s in (s_xs_tmin, s_xs_tmax, s_xs_emin, s_xs_emax, s_xs_dt):
    wire_guarded(_XS_LC, s, refit_xs)
for s in (s_xs_vlo, s_xs_vhi):
    wire_guarded(_XS_LC, s, refresh_xs_lc)
_XS_LC["root"] = VBox([
    widgets.HTML("<b>§7 Step 1 — Cross-spectrum segment</b> (nb × n8, change → refit)"),
    HBox([s_xs_tmin, s_xs_tmax]), HBox([s_xs_emin, s_xs_emax]), s_xs_dt,
    widgets.HTML("<b>Light curves</b> (gold = CS segment)"),
    HBox([s_xs_vlo, s_xs_vhi]), out_xs_lc,
])
clear_output(wait=True)
display(_XS_LC["root"])
refresh_xs_lc()


### §7 Step 2 — Compare nested models (|CS| Leahy)

Toggle (a/b/c) → updates §7 final cell below.


In [10]:
def xs_model_table(active_idx=None):
    active_idx = xs_active_idx() if active_idx is None else int(active_idx)
    rows = []
    for i, nf in enumerate(XS.get("nested_fits") or []):
        row = dict(model=nf["label"], Merit_dof=f"{nf['merit_red']:.2f}", dof=nf["dof"],
                   active="→" if i == active_idx else "")
        if i:
            prev = XS["nested_fits"][i - 1]
            dd, dnu = prev["deviance"] - nf["deviance"], prev["dof"] - nf["dof"]
            p = stats.chi2.sf(max(dd, 0), dnu) if dnu > 0 else np.nan
            row["test"] = f"({chr(96+i-1)})->({chr(96+i)})"
            row["p"] = f"{p:.3g}"
        else:
            row["test"] = row["p"] = "-"
        for k in (1, 2):
            row[f"nu{k}"] = f"{nf[f'nu{k}']:.2f}" if f"nu{k}" in nf else "-"
            row[f"S/N{k}"] = f"{nf[f'sig{k}']:.1f}" if f"sig{k}" in nf else "-"
        rows.append(row)
    return pd.DataFrame(rows) if rows else None


def plot_xs_compare(out, f_lo, f_hi, active_idx=None):
    if not XS.get("nested_fits"):
        return
    active_idx = xs_active_idx() if active_idx is None else int(active_idx)
    mask, ff = xs_pds_band(f_lo, f_hi)
    f, p, err = XS["freq"], XS["power"], XS["leahy_err"]
    fig, ax = plt.subplots(figsize=(8, 4.8))
    ax.errorbar(f[mask], p[mask], xerr=0.5 * XS["ps"].df, yerr=err, fmt="none", alpha=0.35)
    ax.plot(f[mask], p[mask], "o", mfc="none", mec=PDS_COLORS["data"], ms=4, label="|CS| Leahy")
    y_all = [p[mask], [WN_LEVEL]]
    for i, nf in enumerate(XS["nested_fits"]):
        y = nf["model"](ff)
        y_all.append(y[y > 0])
        sel = i == active_idx
        ax.plot(ff, y, color=CMP_COLORS[i], lw=2.4 if sel else 1.2, ls="-" if sel else "--",
                label=f"{nf['label']} ({nf['merit_red']:.2f})" + (" ★" if sel else ""))
    ax.axhline(WN_LEVEL, color=PDS_COLORS["wn"], ls=":", label="WN=2")
    yp = np.concatenate([np.asarray(a)[np.isfinite(a) & (np.asarray(a) > 0)] for a in y_all if np.size(a)])
    ax.set(xscale="log", yscale="log", xlim=(ff[0], ff[-1]),
           ylim=(max(0.8, yp.min() * 0.7), yp.max() * 1.2),
           xlabel="Frequency (Hz)", ylabel="|Cross power| (Leahy)",
           title=f"§7 compare | {XS_TMIN}-{XS_TMAX} s | {XS['nested_fits'][active_idx]['label']}")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.12)
    fig.tight_layout()
    show_in(out, fig)


try:
    _xs_keep = int(s_xs_model.value)
except Exception:
    _xs_keep = int(XS_SELECTED)

_XS_CMP = globals().setdefault("_XS_CMP", dict(gen=0))
dash_teardown(_XS_CMP)
out_xs_tbl, out_xs_pds = widgets.Output(), widgets.Output()

_xs_opts = [(f"({chr(97+i)}) {NESTED_MODEL_LABELS[i].split(')', 1)[-1].strip()}", i)
            for i in range(len(NESTED_MODEL_LABELS))]
s_xs_model = ToggleButtons(options=_xs_opts, value=_xs_keep,
                           description="CS final fit", style={"description_width": "initial"})
s_xs_flo = FloatSlider(value=FMIN_PLOT, min=0.1, max=50, step=0.1, description="f min (Hz)", style=SLIDER_STYLE)
s_xs_fhi = FloatSlider(value=FMAX_PLOT, min=0.1, max=50, step=0.1, description="f max (Hz)", style=SLIDER_STYLE)


def refresh_xs_cmp(_=None):
    idx = xs_active_idx()
    xs_set_active(idx)
    fill_output(out_xs_tbl, xs_model_table(idx))
    plot_xs_compare(out_xs_pds, s_xs_flo.value, s_xs_fhi.value, idx)
    fn = globals().get("refresh_xs_final")
    if callable(fn):
        fn(idx)


def pick_xs_model(change):
    idx = int(change["new"])
    xs_set_active(idx)
    refresh_xs_cmp()
    fn = globals().get("refresh_xs_final")
    if callable(fn):
        fn(idx)


_XS_CMP["widgets"] = (s_xs_model, s_xs_flo, s_xs_fhi)
wire_observer(s_xs_model, pick_xs_model, "xs_cmp_pick")
for s in (s_xs_flo, s_xs_fhi):
    wire_guarded(_XS_CMP, s, refresh_xs_cmp)
_XS_CMP["root"] = VBox([
    widgets.HTML("<b>§7 Step 2 — Nested model comparison (|CS|)</b>"),
    out_xs_tbl,
    widgets.HTML("<b>|Cross power| vs frequency</b> (★ = selected)"),
    HBox([s_xs_flo, s_xs_fhi]), out_xs_pds,
    widgets.HTML("<b>Choose final CS fit</b> (a / b / c) → §7 Step 3 below"),
    s_xs_model,
])
clear_output(wait=True)
display(_XS_CMP["root"])
xs_set_active(_xs_keep)
refresh_xs_cmp()


### §7 Step 3 — Final fit + significance vs single detector

Leahy **|cross power| vs frequency** with components, plus table comparing §6 NaI-b S/N to §7 cross-spectrum S/N and coherence.


In [11]:
def plot_xs_final(out, f_lo, f_hi):
    xfit = XS.get("fit")
    if xfit is None:
        return
    mask, ff = xs_pds_band(f_lo, f_hi)
    f, p, err = XS["freq"], XS["power"], XS["leahy_err"]
    mfit, n_lor = xfit["model"], xfit["n_lor"]
    fig, ax = plt.subplots(figsize=(8, 4.8))
    ax.errorbar(f[mask], p[mask], xerr=0.5 * XS["ps"].df, yerr=err, fmt="none", alpha=0.35)
    ax.plot(f[mask], p[mask], "o", mfc="none", mec=PDS_COLORS["data"], ms=4, label="|CS| data")
    pl = models.PowerLaw1D(mfit.amplitude_0.value, x_0=mfit.x_0_0.value, alpha=mfit.alpha_0.value)
    ax.plot(ff, pl(ff), ":", color=PDS_COLORS["pl"], label=f"PL η₀={xfit['eta0']:.1f}")
    for i in range(n_lor):
        amp = xfit[f"N_lor{i+1}"]
        nu, fwhm = xfit[f"nu{i+1}"], xfit[f"fwhm{i+1}"]
        lor = models.Lorentz1D(amp, nu, fwhm)
        y = lor(ff)
        warn = " (unphysical N<0)" if amp <= 0 else ""
        ax.plot(ff, np.where(y > 0, y, np.nan), "--", color=PDS_COLORS[f"lor{i+1}"],
                label=f"Lor{i+1} {nu:.2f} Hz N={amp:.1f}{warn}")
    ax.plot(ff, mfit(ff), color=PDS_COLORS["total"], lw=2, label="Total")
    ax.axhline(WN_LEVEL, color=PDS_COLORS["wn"], ls=":", label="WN=2")
    y_all = [p[mask], mfit(f[mask]), [WN_LEVEL]]
    yp = np.concatenate([np.asarray(a)[np.isfinite(a) & (np.asarray(a) > 0)] for a in y_all if np.size(a)])
    ax.set(xscale="log", yscale="log", xlim=(ff[0], ff[-1]),
           ylim=(max(0.8, yp.min() * 0.7), yp.max() * 1.2),
           xlabel="Frequency (Hz)", ylabel="|Cross power| (Leahy)",
           title=f"§7 final: {xfit['label']} | Merit/dof={xfit['merit_red']:.2f}")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.12)
    fig.tight_layout()
    show_in(out, fig)


def xs_results_table():
    xfit = XS.get("fit")
    if xfit is None:
        return None
    row = dict(source="Cross-spectrum nb×n8", model=xfit["label"], eta0=xfit["eta0"],
               Merit_dof=f"{xfit['merit_red']:.2f}", dof=xfit["dof"])
    for k in (1, 2):
        if f"nu{k}" in xfit:
            row[f"nu{k}"] = f"{xfit[f'nu{k}']:.2f}"
            row[f"N_lor{k}"] = f"{xfit[f'N_lor{k}']:.1f}"
            row[f"FWHM{k}"] = f"{xfit[f'fwhm{k}']:.2f}"
            row[f"SN{k}"] = f"{xfit[f'sig{k}']:.1f}, {xfit[f'sig{k}_trials']:.1f}"
        else:
            row[f"nu{k}"] = row[f"N_lor{k}"] = row[f"FWHM{k}"] = row[f"SN{k}"] = "-"
    return pd.DataFrame([row])


def refresh_xs_final(idx=None):
    if idx is not None and not isinstance(idx, (int, np.integer)):
        idx = None
    idx = xs_active_idx() if idx is None else int(idx)
    xs_set_active(idx)
    if XS_FINAL.get("out_pds") is None or XS.get("fit") is None:
        return
    xfit = XS["fit"]
    XS_FINAL["hdr"].value = (
        f"<b>§7 Final:</b> <code>{xfit['label']}</code> "
        f"(Merit/dof={xfit['merit_red']:.2f}) | m={XS['cs'].m} segments"
    )
    fill_output(XS_FINAL["out_res"], xs_results_table())
    fill_output(XS_FINAL["out_sig"], significance_table())
    plot_xs_final(XS_FINAL["out_pds"], XS_FINAL["s_flo"].value, XS_FINAL["s_fhi"].value)


XS_FINAL = dict(out_res=None, out_sig=None, out_pds=None, hdr=None, s_flo=None, s_fhi=None)

_XS_ACT = globals().setdefault("_XS_ACT", dict(gen=0))
dash_teardown(_XS_ACT)
out_xs_res, out_xs_sig, out_xs_final = widgets.Output(), widgets.Output(), widgets.Output()
hdr_xs = widgets.HTML("")

s_xs_flo2 = FloatSlider(value=FMIN_PLOT, min=0.1, max=50, step=0.1, description="f min (Hz)", style=SLIDER_STYLE)
s_xs_fhi2 = FloatSlider(value=FMAX_PLOT, min=0.1, max=50, step=0.1, description="f max (Hz)", style=SLIDER_STYLE)

XS_FINAL.update(out_res=out_xs_res, out_sig=out_xs_sig, out_pds=out_xs_final, hdr=hdr_xs,
                s_flo=s_xs_flo2, s_fhi=s_xs_fhi2)

_XS_ACT["widgets"] = (s_xs_flo2, s_xs_fhi2)
wire_guarded(_XS_ACT, s_xs_flo2, refresh_xs_final)
wire_guarded(_XS_ACT, s_xs_fhi2, refresh_xs_final)
if "s_xs_model" in globals():
    wire_observer(s_xs_model, lambda c: refresh_xs_final(int(c["new"])), "xs_act_pick")

_XS_ACT["root"] = VBox([
    hdr_xs,
    widgets.HTML("<b>§7 Step 3 — Final CS fit parameters</b>"),
    out_xs_res,
    widgets.HTML("<b>Significance: individual NaI-b (§6) vs cross-spectrum (§7)</b>"),
    out_xs_sig,
    widgets.HTML("<b>|Cross power| vs frequency — components</b>"),
    HBox([s_xs_flo2, s_xs_fhi2]), out_xs_final,
])
clear_output(wait=True)
display(_XS_ACT["root"])
refresh_xs_final()
